<a href="https://colab.research.google.com/github/romashko1977/skills-github-pages/blob/main/0204026_spiral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, time, warnings, requests
# Автоматическая настройка среды и библиотек
os.system('pip install ccxt pandas_ta --upgrade -q')

import ccxt, pandas_ta as ta, pandas as pd
from IPython.display import display, HTML, clear_output

warnings.filterwarnings('ignore')

class Config:
    # --- ДАННЫЕ ИЗ СКРИНШОТОВ ---
    KEY = 'mx0vglRyG4Ee1fFsfV'
    SECRET = '8c1929b7d096420dadbc416f6fd6035d'
    TG_TOKEN = '8754962684:AAHRdDS30Yfw60TRam2LVoQW4DJ2N5yQA7A'
    TG_USER_ID = '331057418'

    # --- НАСТРОЙКИ СКАЛЬПИНГА ---
    SCAN_LIMIT = 30       # Максимальный охват для поиска выстрелов
    TF = '5m'             # 5 минут — база для импульса
    Z_WINDOW = 12         # Сверхбыстрая реакция на изменение цены
    Z_ENTRY = 3.0         # Ловим только экстремальные "палки"
    RISK_PER_TRADE = 0.05 # 5% от депо на позицию
    LEVERAGE = 20         # Плечо 20х

mexc = ccxt.mexc({'apiKey': Config.KEY, 'secret': Config.SECRET, 'options': {'defaultType': 'swap'}})

def send_tg(msg):
    try: requests.post(f"https://api.telegram.org/bot{Config.TG_TOKEN}/sendMessage",
                       data={"chat_id": Config.TG_USER_ID, "text": msg, "parse_mode": "HTML"}, timeout=5)
    except: pass

def play_meow():
    display(HTML("<script>new Audio('https://cdn.pixabay.com/audio/2022/03/24/audio_3234293f0b.mp3').play();</script>"))

def run_impulse_backtest(df):
    """ Проверка: как часто цена возвращается после такого выстрела """
    z = (df['c'] - df['c'].rolling(Config.Z_WINDOW).mean()) / df['c'].rolling(Config.Z_WINDOW).std()
    res = []
    for i in range(20, len(df)-10):
        side = 'SHORT' if z.iloc[i] > Config.Z_ENTRY else 'LONG' if z.iloc[i] < -Config.Z_ENTRY else None
        if side:
            ent = df['c'].iloc[i]
            tp, sl = (ent*1.008, ent*0.995) if side == 'LONG' else (ent*0.992, ent*1.005)
            for j in range(i+1, i+6): # Ожидаем откат за 30 мин
                if (side=='LONG' and df['h'].iloc[j]>=tp) or (side=='SHORT' and df['l'].iloc[j]<=tp): res.append(1); break
                if (side=='LONG' and df['l'].iloc[j]<=sl) or (side=='SHORT' and df['h'].iloc[j]>=sl): res.append(-1); break
    return (len([x for x in res if x > 0])/len(res)*100) if res else 0, len(res)

def get_trade_data(df, z, bal):
    p = df['c'].iloc[-1]
    atr = ta.atr(df['h'], df['l'], df['c'], 10).iloc[-1]
    side = 'LONG' if z < 0 else 'SHORT'
    # Скальперские уровни на основе волатильности
    tp, sl = (p + atr*2.0, p - atr*1.5) if side == 'LONG' else (p - atr*2.0, p + atr*1.5)
    size_usd = bal * Config.RISK_PER_TRADE * Config.LEVERAGE
    return p, tp, sl, size_usd, abs(tp-p)*(size_usd/p)

ui_html = """
<style>
@keyframes pulse-L { 0%, 100% {background: rgba(0,255,136,0.1);} 50% {background: rgba(0,255,136,0.5);} }
@keyframes pulse-S { 0%, 100% {background: rgba(255,77,77,0.1);} 50% {background: rgba(255,77,77,0.5);} }
.blink-LONG { animation: pulse-L 0.8s infinite; } .blink-SHORT { animation: pulse-S 0.8s infinite; }
</style>
<script>function copyACS(t){const e=document.createElement('textarea');e.value=t;document.body.appendChild(e);e.select();document.execCommand('copy');document.body.removeChild(e);}</script>
"""

active_signals = {}

while True:
    try:
        mexc.load_markets()
        # Стабильное получение баланса
        try:
            bal_raw = mexc.fetch_balance()
            bal_val = float(bal_raw['info']['data'][0]['availableBalance']) if isinstance(bal_raw['info']['data'], list) else float(bal_raw['info']['data']['availableBalance'])
        except: bal_val = 100.0

        symbols = [s for s in mexc.symbols if s.endswith(':USDT')][:Config.SCAN_LIMIT]
        results = []

        for s in symbols:
            try:
                coin = s.split(':')[0]
                df = pd.DataFrame(mexc.fetch_ohlcv(s, Config.TF, limit=150), columns=['ts','o','h','l','c','v'])
                z = ((df['c'] - df['c'].rolling(Config.Z_WINDOW).mean()) / df['c'].rolling(Config.Z_WINDOW).std()).iloc[-1]

                wr, cnt = run_impulse_backtest(df)
                ent, tp, sl, size, prf = get_trade_data(df, z, bal_val)

                old_s = active_signals.get(coin, "WAIT")
                new_s = "LONG" if z < -Config.Z_ENTRY else "SHORT" if z > Config.Z_ENTRY else "WAIT"

                if new_s != "WAIT" and old_s == "WAIT":
                    play_meow()
                    send_tg(f"🚀 <b>IMPULSE: {new_s} {coin}</b>\nSize: ${size:.0f}\nEnt: {ent:.4f}\nTP: {tp:.4f}\nSL: {sl:.4f}\nZ: {z:.2f}")

                active_signals[coin] = new_s
                results.append({'coin':coin,'st':new_s,'z':z,'wr':wr,'cnt':cnt,'ent':ent,'tp':tp,'sl':sl,'size':size,'prf':prf})
            except: continue

        clear_output(wait=True)
        display(HTML(ui_html))
        rows = ""
        # Сортировка: сначала сигналы, потом самые высокие WR
        for r in sorted(results, key=lambda x: (x['st']!="WAIT", x['wr']), reverse=True)[:15]:
            anim = f"blink-{r['st']}" if r['st'] != "WAIT" else ""
            fmt = ".6f" if r['ent'] < 1 else ".4f"
            rows += f"""<tr onclick="copyACS('{r['coin']}')" class='{anim}' style='cursor:pointer; border-bottom:1px solid #222;'>
                <td style='padding:8px;'><b>{r['coin']}</b> 🐾</td>
                <td style='color:{"#0f8" if "LONG" in r['st'] else "#f44" if "SHORT" in r['st'] else "#444"}; font-weight:bold;'>{r['st']}</td>
                <td style='color:#ffa500;'>{r['wr']:.0f}%</td>
                <td>{r['ent']:{fmt}}</td><td style='color:#0f8;'>{r['tp']:{fmt}}</td><td style='color:#f44;'>{r['sl']:{fmt}}</td>
                <td style='color:#00f3ff;'>${r['size']:.0f}</td><td style='color:#0f8;'>${r['prf']:.1f}</td>
                <td style='text-align:right; color:{"#0f8" if abs(r['z'])>Config.Z_ENTRY else "#666"};'>{r['z']:+.2f}</td>
            </tr>"""

        display(HTML(f"""<div style='background:#0b0e11; color:#848e9c; padding:20px; font-family:monospace; border-radius:15px; border:1px solid #1c2242;'>
            <div style='display:flex; justify-content:space-between; align-items:center;'>
                <h2 style='color:#00f3ff; margin:0;'>🛰 ALPHA NEKO v37.4 [SCALP]</h2>
                <b style='color:#0f8; font-size:16px;'>$ {bal_val:.2f}</b>
            </div>
            <table style='width:100%; margin-top:10px; font-size:11px; border-collapse:collapse; text-align:left;'>
                <tr style='color:#fff; border-bottom:2px solid #1c2242;'>
                    <th>COIN</th><th>SIGNAL</th><th>WR%</th><th>ENTRY</th><th>TP</th><th>SL</th><th>SIZE ($)</th><th>PROFIT</th><th style='text-align:right;'>Z-SCR</th>
                </tr>
                {rows}
            </table></div>"""))

    except Exception as e: print(f"Error: {e}"); time.sleep(5)
    time.sleep(12) # Интервал обновления для "ловли" импульсов

COIN,SIGNAL,WR%,ENTRY,TP,SL,SIZE ($),PROFIT,Z-SCR
0G/USDT 🐾,WAIT,100%,0.511200,0.508731,0.513051,$0,$0.0,+1.67
AAPLSTOCK/USDT 🐾,WAIT,100%,256.1500,255.5728,256.5829,$0,$0.0,+2.06
ACS/USDT 🐾,WAIT,100%,0.000169,0.000176,0.000165,$0,$0.0,-0.34
ADBESTOCK/USDT 🐾,WAIT,100%,242.6400,241.8802,243.2098,$0,$0.0,+0.80
1000000BABYDOGE/USDT 🐾,WAIT,0%,0.000373,0.000371,0.000375,$0,$0.0,+0.43
1000000MOG/USDT 🐾,WAIT,0%,0.138600,0.137778,0.139216,$0,$0.0,+1.59
1000BONK/USDT 🐾,WAIT,0%,0.005642,0.005615,0.005662,$0,$0.0,+1.00
1000BTT/USDT 🐾,WAIT,0%,0.000310,0.000311,0.000309,$0,$0.0,-0.78
1000RATS/USDT 🐾,WAIT,0%,0.040100,0.040368,0.039899,$0,$0.0,-0.03
1INCH/USDT 🐾,WAIT,0%,0.087100,0.086534,0.087524,$0,$0.0,+0.48
